# Figures S3–S4 — Tissue-resolved concordance landscapes

Built from the packaged CRISPR concordance table (`cell_line`, `disease` columns).

## Data availability

All inputs for this notebook are **copied into** `For Reviewer/source_data/` (or shown from `illustrations/` when a panel cannot be recomputed).

- No Zenodo download is required.
- No paths outside `For Reviewer/` are used after packaging.
- See `DATA_AVAILABILITY.md` and `source_data/manifest.csv` for origins and checksums.

**Files used below** are listed in each panel section.

In [ ]:
import sys
from pathlib import Path
ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

from linkd_repro import paths, style, io, illustrate
style.apply()
paths.ensure_output_dirs()
print("For Reviewer root:", paths.ROOT)
print("source_data OK:", paths.SOURCE.exists())

## Figure S3 — Affinity–concordance landscapes for canonical oncology genes

In [ ]:

cr = io.read_crispr().dropna(subset=["landmark_correlation", "Target_Affinity"])
genes = cr["Gene"].value_counts().head(25).index
sub = cr[cr["Gene"].isin(genes)]
# facet-like small multiples for first 6 genes
show = list(genes[:6])
fig, axes = plt.subplots(2, 3, figsize=(9, 5.5), sharex=True, sharey=True)
for ax, g in zip(axes.ravel(), show):
    gg = sub[sub["Gene"] == g]
    ax.scatter(gg["landmark_correlation"], gg["Target_Affinity"], s=6, alpha=0.35)
    ax.set_title(g, fontsize=8)
axes[1,0].set_xlabel("concordance")
axes[0,0].set_ylabel("Target_Affinity")
fig.suptitle("Fig S3 — affinity vs concordance")
fig.tight_layout()
out = style.save_panel(fig, "figS3_affinity_concordance", sub[sub["Gene"].isin(show)][["Gene","landmark_correlation","Target_Affinity","Selectivity_Score"]])
plt.show()
print(out)


## Figure S4 — Selectivity–concordance across lineages

In [ ]:

cr = io.read_crispr().dropna(subset=["landmark_correlation", "Selectivity_Score", "disease"])
tissues = cr.groupby("disease")["cell_line"].nunique().sort_values(ascending=False).head(7).index
fig, axes = plt.subplots(2, 4, figsize=(10, 5), sharex=True, sharey=True)
axes = axes.ravel()
for i, t in enumerate(tissues):
    ax = axes[i]
    gg = cr[cr["disease"] == t]
    if len(gg) > 8000:
        gg = gg.sample(8000, random_state=0)
    ax.scatter(gg["landmark_correlation"], gg["Selectivity_Score"], s=3, alpha=0.2)
    ax.set_title(str(t)[:28], fontsize=7)
for j in range(i+1, len(axes)):
    axes[j].axis("off")
fig.suptitle("Fig S4 — selectivity vs concordance by tissue")
fig.tight_layout()
out = style.save_panel(fig, "figS4_tissue_selectivity", cr[cr["disease"].isin(tissues)][["disease","landmark_correlation","Selectivity_Score"]].sample(n=min(30000, len(cr)), random_state=0))
plt.show()
print(out)
